In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [34]:
from langchain.tools import tool
from pprint import pprint

In [16]:
@tool #need to anotate with tool decorator 
def addition(a: float,b: float)-> float:
    """funciton to add two given numbers a + b""" #provide good method name and descritipon and datatype, as it will be used by agent 
    return a + b
@tool
def subtraction(a: float,b: float)-> float:
    """funciton to subtract two given numbers a - b"""
    return a - b

@tool
def square_root(a: float)-> float:
    """funciton to find square root of given numbers sqrt(a) """
    return  a ** 0.5
    

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [18]:
from langchain.agents import create_agent
agent = create_agent(model = model, 
                     tools = [addition,subtraction,square_root], 
                    system_prompt = 'You are an arithemetic wizard. Use tools whenever required')

In [19]:
from langchain.messages import HumanMessage
response = agent.invoke ({'messages' : [HumanMessage('What is 5 + 45')]})
response

{'messages': [HumanMessage(content='What is 5 + 45', additional_kwargs={}, response_metadata={}, id='01d9ce88-805c-45e5-b7a2-1a260247954b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'addition', 'arguments': '{"b": 45, "a": 5}'}, '__gemini_function_call_thought_signatures__': {'a83959e0-d825-4dda-bf80-f52c7c5a0a7f': 'EjQKMgEMOdbHn3BpE5LqM+tioNklLVfhwi138rCSCOYCGJ1tD+8dy7/jNGr+HlYzS+FtOD50'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019daac0-b8bc-7753-b6d3-52de132f4063-0', tool_calls=[{'name': 'addition', 'args': {'b': 45, 'a': 5}, 'id': 'a83959e0-d825-4dda-bf80-f52c7c5a0a7f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 17, 'total_tokens': 203, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='50.0', name='addition', id='41d9a3d8-2b33-44f5-bbca-90a9906c90c2', tool_call_i

In [53]:
from pprint import pprint

for resp in response['messages']:
    print('-' * 50)
    # Print the full message object to see the structure
    # pprint(resp)
    
    # 1. Check if the message has a .tool_calls attribute that is not empty
    if hasattr(resp, 'tool_calls') and resp.tool_calls:
        for tool in resp.tool_calls:
            # Access the tool name and arguments directly from the tool object
            print(f"Tool Name: {tool['name']}")
            print(f"Arguments: {tool['args']}")
            
    # 2. If it's a standard AIMessage with content, print the content
    elif hasattr(resp, 'content') and resp.content:
        print(f"Content: {resp.content}")

--------------------------------------------------
Content: What is 5 + 45
--------------------------------------------------
Tool Name: addition
Arguments: {'b': 45, 'a': 5}
--------------------------------------------------
Content: 50.0
--------------------------------------------------
Content: [{'type': 'text', 'text': '5 + 45 is 50.', 'extras': {'signature': 'EjQKMgEMOdbHLOa4N1Jvi/tzKMYze8f967PFB1e3hDu0qhGwT5NHb0UzJv2Tm75HHnf7ciLS'}}]


In [57]:
from langchain.messages import HumanMessage
response = agent.invoke ({'messages' : [HumanMessage('What is the root of 26')]})
response['messages'][-1].content

[{'type': 'text',
  'text': 'The square root of 26 is approximately 5.099.',
  'extras': {'signature': 'EjQKMgEMOdbHSHj4qX975uQECVQ/X+VsM/X2SR1MzWBWthJlf0tWiOwWgMZdiJTN+wZfK5cx'}}]

In [68]:
from tavily import TavilyClient
from typing import Dict, Any
from langchain.tools import tool

tavily_client = TavilyClient()
#Tavily is a specialized web search engine built for AI agents
# score we see is relevance ranking score
@tool
def web_search(query: str)-> Dict[str, Any]: 
    ''' this method does web search'''
    print('web_search called with query', query)
    return tavily_client.search(query)

In [62]:
web_search.invoke('what is the current date in india')

{'query': 'what is the current date in india',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.timeanddate.com/worldclock/india',
   'title': 'Time in India - Time and Date',
   'content': 'Current Local Time in India ; Anand, Sat 5:59 pm ; Anantapur, Sat 5:59 pm ; Anantnag, Sat 5:59 pm ; Angul, Sat 5:59 pm.',
   'score': 0.9971551,
   'raw_content': None},
  {'url': 'https://24timezones.com/India/time',
   'title': 'Local time in India right now - World Clock',
   'content': 'Exact time and daylight saving time in India in 2026. Current time in IN. 7:57:02 PM, Sunday 19, April 2026 IST',
   'score': 0.99589807,
   'raw_content': None},
  {'url': 'https://www.utctime.net/ist-indian-time-now',
   'title': 'IST Indian Time Now Indian Standard Time - UTC Time',
   'content': 'Indian Standard Time is 5 hours 30 minutes ahead from the UTC universal time. IST Indian current date is 20th Monday April 2026. Current time in IST Indian (IST',
   's

In [64]:
from langchain.messages import HumanMessage
response = agent.invoke ({'messages' : [HumanMessage('What is the current date')]})
response['messages'][-1].content

[{'type': 'text',
  'text': 'The current date is May 22, 2024.',
  'extras': {'signature': 'EjQKMgEMOdbHiSPe+4sNzarCRq//CKMYQh/qiO0MhGH2/j2nEPSbbGS0YsJF2+vzMb8fj6Pb'}}]

In [65]:
agent = create_agent(model = model, tools = [web_search], system_prompt = 'You are an helpful assistant, use tools when needed')

In [70]:
from langchain.messages import HumanMessage
response = agent.invoke ({'messages' : [HumanMessage('What is current date in india')]})
response['messages'][-1].content

[{'type': 'text',
  'text': 'The current date in India is Monday, April 20, 2026.',
  'extras': {'signature': 'EjQKMgEMOdbHN4ZaYr3TCZm8vGgoSyjdAmGeNlvuv4lmDNY6MZVodlTnj/ip5InasVpln2Gr'}}]

In [69]:
from pprint import pprint

for resp in response['messages']:
    print('-' * 50)
    # Print the full message object to see the structure
    # pprint(resp)
    
    # 1. Check if the message has a .tool_calls attribute that is not empty
    if hasattr(resp, 'tool_calls') and resp.tool_calls:
        for tool in resp.tool_calls:
            # Access the tool name and arguments directly from the tool object
            print(f"Tool Name: {tool['name']}")
            print(f"Arguments: {tool['args']}")
            
    # 2. If it's a standard AIMessage with content, print the content
    elif hasattr(resp, 'content') and resp.content:
        print(f"Content: {resp.content}")

--------------------------------------------------
Content: What is current date in india
--------------------------------------------------
Tool Name: web_search
Arguments: {'query': 'current date in India'}
--------------------------------------------------
Content: {"query": "current date in India", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.utctime.net/ist-indian-time-now", "title": "IST Indian Time Now Indian Standard Time - UTC Time", "content": "Indian Standard Time is 5 hours 30 minutes ahead from the UTC universal time. IST Indian current date is 20th Monday April 2026. Current time in IST Indian (IST", "score": 0.7738498, "raw_content": null}, {"url": "https://24timezones.com/India/time", "title": "Local time in India right now - World Clock", "content": "Exact time and daylight saving time in India in 2026. Current time in IN. 7:57:02 PM, Sunday 19, April 2026 IST", "score": 0.7309246, "raw_content": null}, {"url": "https://gr